# Raspberry Model Simulator

This notebook simulates the scattering from raspberry particles based on the model described in the referenced manuscript. You can manually edit the parameters below to explore different configurations.

In [ ]:
# -----------------------------
# Define model parameters
# -----------------------------
Ro_å = 8000.0       # Core radius (Å)
Rp_å = 30.0        # Particle radius (Å)
d = -0.5           # Penetration depth factor (-1 to 1)
Dqo_å = -0.06 # Electron density contrast of oil (e/Å³) - -0.056 for triolein
Dqp_å = 0.12  # Electron density contrast of particles (e/Å³) - 0.12 for proteins
phi_o = 0.2      # Volume fraction of oil
phi_p = 0.0005      # Total volume fraction of particles

phi_ap = 1     # Fraction of particles adsorbed at interface


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import pandas as pd


Ro = Ro_å * 1e-8  # cm
Rp = Rp_å * 1e-8  # cm

re_cm = 2.818e-13  # cm
Dqo = Dqo_å * 1e24 * re_cm  # cm⁻²
Dqp = Dqp_å * 1e24 * re_cm  # cm⁻²



# Calculate surface coverage

# Define q range
# Note: X-ray scattering uses electron density contrast (e/Å³)
q = np.logspace(-4, 0, 2500)
q_cm = q * 1e8
import numpy as np

# Calculate surface coverage

def check_surface_coverage_with_adsorption(Ro, Rp, phi_p, phi_o, phi_ap):
    """
    Computes the surface coverage of adsorbed particles on droplets in a full system.

    Parameters:
    - Ro (float): Core radius in Ångströms.
    - Rp (float): Particle radius in Ångströms.
    - phi_p (float): Total volume fraction of particles.
    - phi_o (float): Volume fraction of oil (droplets).
    - phi_ap (float): Fraction of particles adsorbed at the interface (0 to 1).

    Returns:
    - coverage (float): Fraction of droplet surface area covered by adsorbed particles.
    - warning (str): Message indicating whether the coverage is physically valid.
    """
    import numpy as np

    Vo = (4/3) * np.pi * Ro**3
    Vp = (4/3) * np.pi * Rp**3
    Ao = 4 * np.pi * Ro**2
    Ap = np.pi * Rp**2

    droplet_density = phi_o / Vo
    total_particles = phi_p / Vp

    Np_adsorbed_per_droplet = (total_particles * phi_ap) / droplet_density
    coverage = Np_adsorbed_per_droplet * Ap / Ao

    if coverage > 1.0:
        warning = f"⚠️ Warning: Surface coverage exceeds physical limit! Coverage = {coverage:.2f}"
    else:
        warning = f"✅ Surface coverage is physically valid. Coverage = {coverage:.2f}"

    print(warning)
    return coverage, warning

check_surface_coverage_with_adsorption(Ro, Rp, phi_p, phi_o, phi_ap)

# Helper functions
def W(q_cm, R):
    return 3 * (np.sin(q_cm * R) - q_cm * R * np.cos(q_cm * R)) / (q_cm * R)**3

def Sop(q_cm, Ro, Rp, d):
    return W(q_cm, Ro) * W(q_cm, Rp) * np.sinc(q_cm * (Ro + d * Rp) / np.pi)

def Spp(q_cm, Rp, Ro, d):
    return W(q_cm, Rp)**2 * np.sinc(q_cm * (Ro + d * Rp) / np.pi)**2
    
def compute_scattering(q_cm, Ro, Rp, d, Dqo, Dqp, phi_o, phi_p, phi_ap):
    Vo = (4/3) * np.pi * Ro**3
    Vp = (4/3) * np.pi * Rp**3

    # Surface coverage and number of particles per droplet
    v = (4 * phi_p * phi_ap * (Ro + d * Rp)) / (phi_o * Rp)
    Np = (phi_p * phi_ap * Vo) / (phi_o * Vp)

    # Total scattering length of the complex
    M = Dqo * Vo + Np * Dqp * Vp #(Eq. 11)

    # Normalized per-complex form factor (Eq. 10)
    P_op = (1 / M**2) * (
        (Dqo**2) * Vo**2 * W(q_cm, Ro)**2 +
        Np * (Dqp**2) * Vp**2 * W(q_cm, Rp)**2 +
        Np * (Np - 1) * (Dqp**2) * Vp**2 * Spp(q_cm, Rp, Ro, d) +
        2 * Np * Dqo * Dqp * Vo * Vp * Sop(q_cm, Ro, Rp, d)
    )

    # Number density of droplets
    n_droplets = phi_o / Vo

    # Intensity from droplet–particle complexes

    
    I_op = P_op  * (phi_o * (Dqo**2) * Vo + phi_p * phi_ap * Np * (Dqp**2)*Vp)
    
    # Intensity from free particle contribution
    I_p = (phi_p * (1 - phi_ap)) * (Dqp**2) * Vp * W(q_cm, Rp)**2


    # Total intensity
    I_total = I_op + I_p # (Eq 12)

    return I_op, I_p, I_total


# Compute scattering
I_op, I_p, I_total = compute_scattering(q_cm, Ro, Rp, d, Dqo, Dqp, phi_o, phi_p, phi_ap)

# Apply smearing

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
import numpy as np

def smear_in_log_q_space(q, I_total, sigma, background):
    """
    Applies Gaussian smoothing in log(q) space.

    Parameters:
    - q (np.ndarray): Original q values (log-spaced).
    - I_total (np.ndarray): Intensity values.
    - sigma (float): Smoothing width in log(q) space.
    - background (float): Constant background to avoid log(0).

    Returns:
    - I_smeared (np.ndarray): Smeared intensity values at original q points.
    """
    # Step 1: Add background to avoid log(0)
    I_total_bg = np.where(I_total + background <= 0, background, I_total + background)

    # Step 2: Create uniform log(q) grid
    log_q = np.log10(q)
    log_q_uniform = np.linspace(log_q.min(), log_q.max(), len(q))

    # Step 3: Interpolate intensity onto uniform log(q) grid
    interp_to_uniform = interp1d(log_q, I_total_bg, kind='linear', fill_value="extrapolate")
    I_uniform = interp_to_uniform(log_q_uniform)

    # Step 4: Apply Gaussian smoothing
    I_smooth_uniform = gaussian_filter1d(I_uniform, sigma=sigma)

    # Step 5: Interpolate back to original q grid
    interp_back = interp1d(log_q_uniform, I_smooth_uniform, kind='linear', fill_value="extrapolate")
    I_smeared = interp_back(log_q)

    return I_smeared


I_smeared = smear_in_log_q_space(q, I_total, sigma=5, background=1e-6)

# Compute Porod plot
porod_y = q**-4 * I_total
log_q = np.log10(q)

# Plotting
plt.figure(figsize=(12, 8))

# Plot 1: Complex, Free Particle, and Total Scattering (log-log)
plt.subplot(2, 2, 1)
plt.loglog(q, I_op, label='Complex Contribution (I_op)')
plt.loglog(q, I_p, label='Free Particle Contribution (I_p)')
plt.loglog(q, I_total, label='Total Scattering')
plt.xlabel('q (Å⁻¹)')
plt.ylabel('Intensity (a.u.)')
plt.title('Scattering Contributions')
plt.legend()
plt.grid(True)



# Export scattering components to CSV
df_scattering = pd.DataFrame({
    'q': q,
    'I_op': I_op,
    'I_p': I_p,
    'I_total': I_total
})
df_scattering.to_csv('scattering_components.csv', index=False)


# Plot 2: Smeared Total Scattering (log-log)
plt.subplot(2, 2, 2)
plt.loglog(q, I_smeared, label='Smeared Total Scattering', color='purple')
plt.xlabel('q (Å⁻¹)')
plt.ylabel('Intensity (a.u.)')
plt.title('Smeared Scattering')
plt.grid(True)

                  
# Plot 3: Porod Plot
def plot_porod_fixed_slope(q, I_smeared, q_0):
    """
    Plot Porod region using smeared intensity with fixed slope = -4 in log–log space.

    Parameters:
    - q: numpy array of q-values [Å⁻¹]
    - I_smeared: numpy array of smeared intensity values
    - q_0: specific q-value to use for fitting the intercept
    """
    import numpy as np
    import matplotlib.pyplot as plt

    # Select high-q Porod region
    q_porod_min = q_0  # Use q_0 as the minimum for the Porod region
    mask = q > q_porod_min
    q_fit = q[mask]
    I_fit = I_smeared[mask]

    # Fit intercept using a single point at q_0
    idx_q0 = (np.abs(q - q_0)).argmin()
    I_q0 = I_smeared[idx_q0]
    A = I_q0 / q[idx_q0]**(-4)
    intercept = np.log10(A)
    I_fitted = A * q_fit**(-4)

    # Plot 1: I(q) * q^4
    plt.figure(figsize=(7, 5))
    plt.loglog(q_fit, I_fit * q_fit**4, label=r'$I(q) \cdot q^4$', color='blue')
    plt.axhline(np.mean(I_fit * q_fit**4), color='gray', linestyle='--', label='Mean level')
    plt.xlabel(r'$q$ [$\mathrm{\AA}^{-1}$]')
    plt.ylabel(r'$I(q) \cdot q^4$')
    plt.title('Porod Plot (Fixed Slope $q^{-4}$)')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # Plot 2: log-log I(q) with fixed slope fit
    plt.figure(figsize=(7, 5))
    plt.loglog(q, I_smeared, label=r'$I(q)$ (Smeared)', color='darkgreen')
    plt.loglog(q_fit, I_fitted,
               label=fr'Fixed Fit: $I \propto q^{{-4}}$, Intercept = {intercept:.2f}', linestyle='dotted')
    plt.xlabel(r'$q$ [$\mathrm{\AA}^{-1}$]')
    plt.ylabel(r'$I(q)$ [a.u.]')
    plt.title('Porod Region Fit (Fixed Slope $q^{-4}$)')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    return q_fit, I_fitted
q_fit, I_fitted = plot_porod_fixed_slope(q, I_smeared, q_0=0.012)


# Export smeared intensity to CSV
df_smeared = pd.DataFrame({'q': q,'I_smeared': I_smeared})
df_smeared.to_csv('smeared_intensity.csv', index=False)

# Export q-4 intensity to CSV
df_porod_fit = pd.DataFrame({'q_fit': q_fit,'I_fitted': I_fitted})
df_porod_fit.to_csv('slope_intensity.csv', index=False)